# Milestone 2 — Data Cleaning
## AI Bill Anomaly Detection System

Our synthetic dataset from Milestone 1 is already fairly clean (no nulls, consistent dtypes),
which isn't realistic — real invoice data is messy. So first we **inject realistic noise**
(whitespace, inconsistent casing, a few missing values, a couple of accidental exact
duplicates, and mixed date formats), then clean it properly. This gives the cleaning step
actual work to do and mirrors what you'd hit with a real dataset.

**Critical rule for this project:** we clean *noise*, not *signal*. The anomalies we
deliberately injected in Milestone 1 (`True_Anomaly == 1`) — high amounts, invalid GST,
duplicate invoices, high-frequency vendors — must **not** be removed or capped here. Only
genuine data-entry mess gets cleaned.

In [1]:
import pandas as pd
import numpy as np
import random

np.random.seed(7)
random.seed(7)

df = pd.read_csv("../data/raw/invoices.csv")
print("Loaded shape:", df.shape)
df.head()

Loaded shape: (10200, 9)


,Invoice_ID,Vendor_Name,Department,Invoice_Date,Amount,GST,Quantity,Category,True_Anomaly
0,INV200078,Vendor_030,Health,2024-01-13,10359.12,12,16,Vehicle Maintenance,1
1,INV103742,Vendor_031,Administration,2025-04-29,17481.48,5,5,Vehicle Maintenance,0
2,INV200068,Vendor_024,Water Supply,2025-04-17,21126.69,5,23,Electronics,1
3,INV105723,Vendor_039,Transport,2025-07-15,11991.82,5,33,Office Equipment,0
4,INV108987,Vendor_037,Transport,2024-07-12,13688.73,12,9,Construction Material,0


## Step 1: Simulate realistic messiness (for demonstration)

In [2]:
df_messy = df.copy()

# 1. Inconsistent whitespace / casing in text fields
whitespace_idx = np.random.choice(df_messy.index, 150, replace=False)
df_messy.loc[whitespace_idx, "Vendor_Name"] = df_messy.loc[whitespace_idx, "Vendor_Name"].apply(
    lambda x: f"  {x.lower()}  "
)

case_idx = np.random.choice(df_messy.index, 150, replace=False)
df_messy.loc[case_idx, "Category"] = df_messy.loc[case_idx, "Category"].str.upper()

# 2. A few missing values (accidental, not the anomaly logic from Milestone 1)
missing_gst_idx = np.random.choice(df_messy.index, 40, replace=False)
df_messy.loc[missing_gst_idx, "GST"] = np.nan

missing_qty_idx = np.random.choice(df_messy.index, 25, replace=False)
df_messy.loc[missing_qty_idx, "Quantity"] = np.nan

# 3. Mixed date formats (common in real exports)
mixed_date_idx = np.random.choice(df_messy.index, 200, replace=False)
df_messy.loc[mixed_date_idx, "Invoice_Date"] = pd.to_datetime(
    df_messy.loc[mixed_date_idx, "Invoice_Date"]
).dt.strftime("%d/%m/%Y")

# 4. A handful of accidental EXACT duplicate rows (true data-entry duplicates,
#    different from the intentional "duplicate invoice" anomaly pattern from Milestone 1)
exact_dup_rows = df_messy.sample(15, random_state=1)
df_messy = pd.concat([df_messy, exact_dup_rows], ignore_index=True)

print("Messy dataset shape:", df_messy.shape)
df_messy.sample(5)

Messy dataset shape: (10215, 9)


,Invoice_ID,Vendor_Name,Department,Invoice_Date,Amount,GST,Quantity,Category,True_Anomaly
7358,INV105773,Vendor_002,Sanitation,2024-10-02,9992.49,12.0,26.0,Cleaning Supplies,0
10038,INV105530,Vendor_023,Health,2024-11-06,9952.62,28.0,48.0,Office Equipment,0
6597,INV102101,Vendor_015,Health,2025-04-18,11019.26,5.0,26.0,Software License,0
8105,INV108975,Vendor_014,Transport,2025-08-16,13878.28,5.0,1.0,Construction Material,0
10086,INV101028,Vendor_006,Water Supply,2024-12-12,19257.32,5.0,48.0,Office Equipment,0


## Step 2: Initial inspection

In [3]:
print("Missing values per column:")
print(df_messy.isnull().sum())
print("\nDtypes:")
print(df_messy.dtypes)
print("\nExact duplicate rows (all columns identical):", df_messy.duplicated().sum())
print("Duplicate Invoice_IDs:", df_messy['Invoice_ID'].duplicated().sum())

Missing values per column:
Invoice_ID       0
Vendor_Name      0
Department       0
Invoice_Date     0
Amount           0
GST             40
Quantity        25
Category         0
True_Anomaly     0
dtype: int64

Dtypes:
Invoice_ID          str
Vendor_Name         str
Department          str
Invoice_Date        str
Amount          float64
GST             float64
Quantity        float64
Category            str
True_Anomaly      int64
dtype: object

Exact duplicate rows (all columns identical): 15
Duplicate Invoice_IDs: 15


## Step 3: Remove accidental exact duplicates

We only drop rows that are **fully identical across every column, including Invoice_ID**.
These are true data-entry duplicates. The Milestone-1 "duplicate invoice" anomaly rows have
**different Invoice_IDs**, so they are correctly left untouched — they're a fraud signal,
not a cleaning issue.

In [4]:
before = len(df_messy)
df_clean = df_messy.drop_duplicates(keep="first").reset_index(drop=True)
after = len(df_clean)

print(f"Removed {before - after} exact duplicate rows")
print("Remaining shape:", df_clean.shape)

Removed 15 exact duplicate rows
Remaining shape: (10200, 9)


## Step 4: Handle missing values

- `GST` missing → fill with the **department's most common GST rate** (mode), since GST
  slabs are standardized and department purchasing patterns tend to repeat.
- `Quantity` missing → fill with the **category median**, since quantity varies a lot by
  item type but is more stable within a category.

In [5]:
print("Missing before:\n", df_clean[["GST", "Quantity"]].isnull().sum())

df_clean["GST"] = df_clean.groupby("Department")["GST"].transform(
    lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else x.median())
)

df_clean["Quantity"] = df_clean.groupby("Category")["Quantity"].transform(
    lambda x: x.fillna(x.median())
)

print("\nMissing after:\n", df_clean[["GST", "Quantity"]].isnull().sum())

Missing before:
 GST         40
Quantity    25
dtype: int64

Missing after:
 GST         0
Quantity    0
dtype: int64


## Step 5: Standardize text fields

In [6]:
df_clean["Vendor_Name"] = df_clean["Vendor_Name"].str.strip().str.title()
df_clean["Category"] = df_clean["Category"].str.strip().str.title()
df_clean["Department"] = df_clean["Department"].str.strip().str.title()

# Quick check: vendor names should now be consistent (no case/whitespace variants
# being treated as different vendors)
print("Unique vendors after standardization:", df_clean["Vendor_Name"].nunique())
df_clean[["Vendor_Name", "Category", "Department"]].sample(5)

Unique vendors after standardization: 42


,Vendor_Name,Category,Department
4764,Vendor_039,Furniture,Water Supply
6168,Vendor_034,Fuel,Public Works
628,Vendor_026,Office Equipment,Public Works
4500,Vendor_033,Vehicle Maintenance,Health
9651,Vendor_019,Cleaning Supplies,Sanitation


## Step 6: Fix date format and data types

In [7]:
# Real invoice exports often mix formats (YYYY-MM-DD from one system, DD/MM/YYYY from
# another). We parse in two passes: first the default (ISO) format, then retry any
# failures with dayfirst=True to catch the DD/MM/YYYY rows.
parsed_default = pd.to_datetime(df_clean["Invoice_Date"], errors="coerce")
unparsed_mask = parsed_default.isnull()
unparsed = unparsed_mask.sum()

if unparsed > 0:
    print(f"{unparsed} dates unparsed on first pass, retrying with dayfirst=True")
    retried = pd.to_datetime(df_clean.loc[unparsed_mask, "Invoice_Date"], dayfirst=True, errors="coerce")
    parsed_default.loc[unparsed_mask] = retried

df_clean["Invoice_Date"] = parsed_default

still_unparsed = df_clean["Invoice_Date"].isnull().sum()
print(f"Remaining unparsed dates after retry: {still_unparsed}")

df_clean["Amount"] = pd.to_numeric(df_clean["Amount"], errors="coerce")
df_clean["GST"] = pd.to_numeric(df_clean["GST"], errors="coerce")
df_clean["Quantity"] = pd.to_numeric(df_clean["Quantity"], errors="coerce").astype(int)
df_clean["True_Anomaly"] = df_clean["True_Anomaly"].astype(int)

print("\nFinal dtypes:")
print(df_clean.dtypes)
print("\nAny remaining nulls after type conversion:")
print(df_clean.isnull().sum())

200 dates unparsed on first pass, retrying with dayfirst=True
Remaining unparsed dates after retry: 0

Final dtypes:
Invoice_ID                 str
Vendor_Name                str
Department                 str
Invoice_Date    datetime64[us]
Amount                 float64
GST                    float64
Quantity                 int64
Category                   str
True_Anomaly             int64
dtype: object

Any remaining nulls after type conversion:
Invoice_ID      0
Vendor_Name     0
Department      0
Invoice_Date    0
Amount          0
GST             0
Quantity        0
Category        0
True_Anomaly    0
dtype: int64


## Step 7: Outlier identification (NOT removal)

We flag statistical outliers in `Amount` using the IQR method — purely for **visibility in
EDA (Milestone 3)**. We do **not** drop or cap them, because a meaningful share of these
outliers are exactly the anomalies this whole project exists to detect. Removing them here
would destroy the model's training signal.

In [8]:
Q1 = df_clean["Amount"].quantile(0.25)
Q3 = df_clean["Amount"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_clean["IQR_Outlier_Flag"] = (
    (df_clean["Amount"] < lower_bound) | (df_clean["Amount"] > upper_bound)
).astype(int)

print(f"IQR bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Statistical outliers flagged: {df_clean['IQR_Outlier_Flag'].sum()} "
      f"({df_clean['IQR_Outlier_Flag'].mean()*100:.2f}%)")

# Sanity check: how much overlap is there between IQR outliers and our known injected anomalies?
overlap = df_clean[(df_clean["IQR_Outlier_Flag"] == 1) & (df_clean["True_Anomaly"] == 1)]
print(f"Overlap between IQR outliers and true anomalies: {len(overlap)} rows")

IQR bounds: [-3183.16, 35137.18]
Statistical outliers flagged: 300 (2.94%)
Overlap between IQR outliers and true anomalies: 300 rows


## Step 8: Final validation before saving

In [9]:
print("Final cleaned dataset shape:", df_clean.shape)
print("\nMissing values:\n", df_clean.isnull().sum())
print("\nDtypes:\n", df_clean.dtypes)
print("\nAnomaly distribution preserved:")
print(df_clean["True_Anomaly"].value_counts())
print(f"Anomaly rate: {df_clean['True_Anomaly'].mean()*100:.2f}%")

df_clean.describe(include="all").T

Final cleaned dataset shape: (10200, 10)

Missing values:
 Invoice_ID          0
Vendor_Name         0
Department          0
Invoice_Date        0
Amount              0
GST                 0
Quantity            0
Category            0
True_Anomaly        0
IQR_Outlier_Flag    0
dtype: int64

Dtypes:
 Invoice_ID                     str
Vendor_Name                    str
Department                     str
Invoice_Date        datetime64[us]
Amount                     float64
GST                        float64
Quantity                     int64
Category                       str
True_Anomaly                 int64
IQR_Outlier_Flag             int64
dtype: object

Anomaly distribution preserved:
True_Anomaly
0    9358
1     842
Name: count, dtype: int64
Anomaly rate: 8.25%


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
Invoice_ID,10200,10200,INV200078,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Vendor_Name,10200,42,Vendor_030,296,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Department,10200,8,Administration,1359,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Invoice_Date,10200,NaN,NaN,NaN,2025-01-01 12:55:54.352941,2024-01-01 00:00:00,2024-07-05 00:00:00,2025-01-03 00:00:00,2025-07-06 00:00:00,2025-12-31 00:00:00,NaN
Amount,10200.0,NaN,NaN,NaN,19458.462288,642.56,11186.965,16206.955,20767.05,248768.32,23410.715863
GST,10200.0,NaN,NaN,NaN,15.961176,1.0,5.0,18.0,28.0,55.0,9.036388
Quantity,10200.0,NaN,NaN,NaN,25.009118,1.0,13.0,25.0,37.0,49.0,14.111855
Category,10200,10,Construction Material,1079,NaN,NaN,NaN,NaN,NaN,NaN,NaN
True_Anomaly,10200.0,NaN,NaN,NaN,0.082549,0.0,0.0,0.0,0.0,1.0,0.275213
IQR_Outlier_Flag,10200.0,NaN,NaN,NaN,0.029412,0.0,0.0,0.0,0.0,1.0,0.168966


In [10]:
import os
os.makedirs("../data/processed", exist_ok=True)
output_path = "../data/processed/invoices_cleaned.csv"
df_clean.to_csv(output_path, index=False)
print(f"Saved cleaned dataset to {output_path}")
print(f"Final rows: {len(df_clean)}, columns: {len(df_clean.columns)}")

Saved cleaned dataset to ../data/processed/invoices_cleaned.csv
Final rows: 10200, columns: 10


## Summary — What this notebook did

| Step | Action | Rows/values affected |
|---|---|---|
| Duplicates | Removed exact accidental duplicates only | 15 removed |
| Missing GST | Filled with department mode | 40 filled |
| Missing Quantity | Filled with category median | 25 filled |
| Text standardization | Trimmed + title-cased Vendor/Category/Department | ~300 rows |
| Date parsing | Unified mixed date formats to datetime | 200 rows |
| Dtype correction | Amount/GST → float, Quantity → int | all rows |
| Outlier flagging | IQR method, flagged only, not removed | see output above |

**Important:** `True_Anomaly` count is unchanged from Milestone 1 — cleaning did not
touch the anomaly signal, only genuine data-quality noise.

